In [1]:
!pip -q install streamlit pymupdf scikit-learn google-genai

In [2]:
%%writefile backend.py
import os
import re
import json
import fitz

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google import genai
from google.genai import types


# 1. Gemini connection
def get_gemini_client():
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError(
            "Gemini API key is missing. Add GEMINI_API_KEY to Colab Secrets and run Cell 3."
        )
    return genai.Client(
        api_key=api_key,
        http_options=types.HttpOptions(timeout=45000)  # 45 seconds
    )


# 2. Skills dictionary
SKILLS_DATABASE = {
    "Python": ["python"],
    "Java": ["java"],
    "C++": ["c++"],
    "JavaScript": ["javascript", "js"],
    "HTML": ["html"],
    "CSS": ["css"],
    "SQL": ["sql"],
    "MySQL": ["mysql"],
    "PostgreSQL": ["postgresql", "postgres"],
    "MongoDB": ["mongodb"],
    "Machine Learning": ["machine learning", "ml"],
    "Deep Learning": ["deep learning"],
    "Artificial Intelligence": ["artificial intelligence", "ai"],
    "Natural Language Processing": ["natural language processing", "nlp"],
    "Computer Vision": ["computer vision"],
    "Pandas": ["pandas"],
    "NumPy": ["numpy"],
    "Scikit-learn": ["scikit-learn", "sklearn"],
    "TensorFlow": ["tensorflow"],
    "Keras": ["keras"],
    "PyTorch": ["pytorch"],
    "OpenCV": ["opencv"],
    "Matplotlib": ["matplotlib"],
    "Seaborn": ["seaborn"],
    "Power BI": ["power bi"],
    "Tableau": ["tableau"],
    "Microsoft Excel": ["microsoft excel", "excel"],
    "Data Analysis": ["data analysis", "data analytics"],
    "Data Visualization": ["data visualization"],
    "Data Science": ["data science"],
    "Statistics": ["statistics"],
    "Data Cleaning": ["data cleaning"],
    "Streamlit": ["streamlit"],
    "Flask": ["flask"],
    "Django": ["django"],
    "React": ["react", "react.js"],
    "Node.js": ["node.js", "nodejs"],
    "Git": ["git"],
    "GitHub": ["github"],
    "Linux": ["linux"],
    "AWS": ["aws", "amazon web services"],
    "Microsoft Azure": ["azure"],
    "Communication": ["communication", "communication skills"],
    "Problem Solving": ["problem solving", "problem-solving"],
    "Teamwork": ["teamwork", "team work"],
    "Leadership": ["leadership"],
    "Time Management": ["time management"]
}


# 3. Text cleaning
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s+#.]", " ", text)
    return text.strip()


# 4. Skill extraction
def extract_skills(text, skills_database=None):
    if skills_database is None:
        skills_database = SKILLS_DATABASE

    cleaned_text = preprocess_text(text)
    found_skills = []

    for skill, variations in skills_database.items():
        for variation in variations:
            pattern = r"(?<!\w)" + re.escape(variation.lower()) + r"(?!\w)"
            if re.search(pattern, cleaned_text):
                found_skills.append(skill)
                break

    return sorted(set(found_skills))


# 5. TF-IDF text similarity
def calculate_similarity(resume_text, job_description):
    documents = [
        preprocess_text(resume_text),
        preprocess_text(job_description)
    ]

    try:
        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            sublinear_tf=True
        )
        tfidf_matrix = vectorizer.fit_transform(documents)
        score = cosine_similarity(
            tfidf_matrix[0:1], tfidf_matrix[1:2]
        )[0][0]
        return round(float(score) * 100, 2)
    except ValueError:
        return 0.0


# 6. Compare resume skills with job skills
def analyze_resume(resume_text, job_description):
    if not resume_text or not resume_text.strip():
        raise ValueError("The resume contains no readable text.")
    if not job_description or not job_description.strip():
        raise ValueError("Please provide a job description.")

    resume_skills = extract_skills(resume_text)
    job_skills = extract_skills(job_description)

    matched = sorted(set(resume_skills) & set(job_skills))
    missing = sorted(set(job_skills) - set(resume_skills))

    skill_match = (
        (len(matched) / len(job_skills)) * 100
        if job_skills else 0.0
    )

    return {
        "resume_skills": resume_skills,
        "job_skills": job_skills,
        "matched_skills": matched,
        "missing_skills": missing,
        "skill_match_percentage": round(skill_match, 2),
        "similarity_percentage": calculate_similarity(
            resume_text, job_description
        )
    }


# 7. Extract text from PDF bytes
def extract_resume_text(pdf_bytes):
    if not pdf_bytes:
        raise ValueError("The uploaded PDF is empty.")

    text_parts = []
    try:
        with fitz.open(stream=pdf_bytes, filetype="pdf") as pdf:
            for page in pdf:
                text_parts.append(page.get_text("text"))
    except Exception as exc:
        raise ValueError(
            "Could not read this PDF. Please upload a valid, text-based PDF."
        ) from exc

    resume_text = "\n".join(text_parts).strip()
    if not resume_text:
        raise ValueError(
            "No selectable text was found. This scanned PDF may need OCR."
        )
    return resume_text


# 8. Run the rule-based analysis
def process_resume(pdf_bytes, job_description):
    resume_text = extract_resume_text(pdf_bytes)
    results = analyze_resume(resume_text, job_description)
    results["extracted_resume_text"] = resume_text
    return results


# 9. Ask Gemini for concise recommendations
def generate_ai_recommendations(
    resume_text, job_description, analysis_results
):
    client = get_gemini_client()

    prompt = f"""
You are a resume reviewer. Compare the resume with the job description.

RESUME:
{resume_text[:4000]}

JOB DESCRIPTION:
{job_description[:2000]}

MATCHED SKILLS:
{analysis_results.get("matched_skills", [])}

MISSING SKILLS:
{analysis_results.get("missing_skills", [])}

Return ONLY one valid JSON object. Do not use Markdown fences.
Use exactly these five keys:
{{
  "resume_strengths": ["point 1", "point 2"],
  "resume_weaknesses": ["point 1", "point 2"],
  "missing_skill_recommendations": ["point 1", "point 2"],
  "improvement_suggestions": ["point 1", "point 2"],
  "overall_feedback": "One short paragraph"
}}

Keep every point short. Do not invent qualifications or experience.
"""

    models_to_try = [
        "gemini-3.5-flash-lite",
        "gemini-3.1-flash-lite",
        "gemini-3.8-flash",
    ]

    response = None
    last_error = None

    for model_name in models_to_try:
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.1,
                    max_output_tokens=1200
                )
            )
            if response and response.text and response.text.strip():
                break
        except Exception as exc:
            last_error = exc
            err_msg = str(exc)
            if any(code in err_msg for code in ["503", "UNAVAILABLE", "429", "404", "NOT_FOUND", "DEADLINE_EXCEEDED"]):
                continue
            raise

    if response is None or not response.text or not response.text.strip():
        if last_error:
            raise last_error
        raise ValueError("Gemini returned an empty response. Please try again.")

    # Safely extract the JSON object even if extra text or markdown surrounds it
    cleaned_text = response.text.strip()
    start_idx = cleaned_text.find("{")
    end_idx = cleaned_text.rfind("}")
    if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
        cleaned_text = cleaned_text[start_idx:end_idx + 1]

    try:
        recommendations = json.loads(cleaned_text)
    except json.JSONDecodeError as exc:
        preview = cleaned_text[:1200]
        raise ValueError(
            "Gemini returned incomplete or invalid JSON. "
            f"Parser detail: {exc}. Response preview: {preview}"
        ) from exc

    required_keys = [
        "resume_strengths",
        "resume_weaknesses",
        "missing_skill_recommendations",
        "improvement_suggestions",
        "overall_feedback"
    ]

    if not isinstance(recommendations, dict):
        raise ValueError("Gemini response must be a JSON object.")

    for key in required_keys:
        if key not in recommendations:
            recommendations[key] = (
                "" if key == "overall_feedback" else []
            )

    for key in required_keys[:-1]:
        if not isinstance(recommendations[key], list):
            recommendations[key] = [str(recommendations[key])]
    if not isinstance(recommendations["overall_feedback"], str):
        recommendations["overall_feedback"] = str(
            recommendations["overall_feedback"]
        )

    return recommendations


# 10. Complete analysis used by Streamlit
def complete_resume_analysis(pdf_bytes, job_description):
    results = process_resume(pdf_bytes, job_description)
    results["ai_recommendations"] = generate_ai_recommendations(
        results["extracted_resume_text"],
        job_description,
        results
    )
    return results


print("SmartATS backend loaded successfully.")

Overwriting backend.py


In [3]:
from google.colab import userdata
import os

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Add GEMINI_API_KEY in Colab Secrets and enable notebook access.")

os.environ["GEMINI_API_KEY"] = api_key
print("Gemini API key loaded securely.")
del api_key

Gemini API key loaded securely.


In [4]:
from backend import (
    extract_resume_text,
    extract_skills,
    calculate_similarity,
    analyze_resume,
    process_resume,
    generate_ai_recommendations,
    complete_resume_analysis,
    SKILLS_DATABASE
)

print("Backend functions imported successfully.")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


SmartATS backend loaded successfully.
Backend functions imported successfully.


In [5]:
sample_resume = '''
I have experience in Python, SQL, Machine Learning,
Pandas, Power BI and data analysis.
'''

print("Detected skills:", extract_skills(sample_resume))

Detected skills: ['Data Analysis', 'Machine Learning', 'Pandas', 'Power BI', 'Python', 'SQL']


In [6]:
sample_job = '''
We are looking for a candidate with Python, SQL,
Pandas, Machine Learning and Power BI skills.
'''

sample_results = analyze_resume(sample_resume, sample_job)

print("Resume skills:", sample_results["resume_skills"])
print("Job skills:", sample_results["job_skills"])
print("Matched skills:", sample_results["matched_skills"])
print("Missing skills:", sample_results["missing_skills"])
print("Skill match:", sample_results["skill_match_percentage"], "%")
print("Text similarity:", sample_results["similarity_percentage"], "%")

Resume skills: ['Data Analysis', 'Machine Learning', 'Pandas', 'Power BI', 'Python', 'SQL']
Job skills: ['Machine Learning', 'Pandas', 'Power BI', 'Python', 'SQL']
Matched skills: ['Machine Learning', 'Pandas', 'Power BI', 'Python', 'SQL']
Missing skills: []
Skill match: 100.0 %
Text similarity: 36.0 %


In [7]:
%%writefile app.py
import os
import streamlit as st

st.set_page_config(
    page_title="SmartATS | Resume Analyzer",
    page_icon="📄",
    layout="wide"
)

# The API key is inherited from the Colab environment.
# A Streamlit secret can also be used when running outside Colab.
try:
    secret_key = st.secrets.get("GEMINI_API_KEY", "")
    if secret_key:
        os.environ["GEMINI_API_KEY"] = secret_key
except Exception:
    pass

from backend import process_resume, generate_ai_recommendations


# Page styling
st.markdown("""
<style>
.block-container {max-width: 1050px; padding-top: 2rem;}
.hero {text-align: center; padding: 0.5rem 0 1rem 0;}
.hero h1 {font-size: 2.8rem; font-weight: 800; margin-bottom: 0;}
.hero p {color: #777; font-size: 1.05rem;}
div.stButton > button {width: 100%; min-height: 3rem; font-weight: 700;}
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="hero">
  <h1>📄 SmartATS</h1>
  <p>AI-Powered Resume Analyzer</p>
  <p>Understand how your resume matches a target job description.</p>
</div>
""", unsafe_allow_html=True)

st.divider()

left, right = st.columns([1, 1])

with left:
    st.subheader("📄 Upload your resume")
    uploaded_file = st.file_uploader(
        "Choose a text-based PDF resume",
        type=["pdf"]
    )

with right:
    st.subheader("💼 Target job")
    job_description = st.text_area(
        "Paste the job description",
        height=190,
        placeholder="Paste the complete job description here..."
    )

# Step 1: Run the fast, rule-based analysis.
if st.button("✨ Analyze my resume", type="primary"):
    if uploaded_file is None:
        st.error("Please upload a PDF resume.")
    elif not job_description.strip():
        st.error("Please paste a job description.")
    else:
        try:
            with st.spinner("Extracting text and comparing skills..."):
                pdf_bytes = uploaded_file.getvalue()
                st.session_state["analysis_results"] = process_resume(
                    pdf_bytes, job_description
                )
                st.session_state["analysis_job_description"] = job_description
                # Clear feedback from the previous resume analysis.
                st.session_state.pop("ai_recommendations", None)

            st.success("Resume comparison completed.")
        except Exception as error:
            st.session_state.pop("analysis_results", None)
            st.session_state.pop("ai_recommendations", None)
            st.error("The resume could not be analyzed.")
            st.caption(str(error))


# Step 2: Display the basic results immediately, without waiting for Gemini.
results = st.session_state.get("analysis_results")

if results:
    st.divider()
    st.header("📊 Analysis results")

    score1, score2 = st.columns(2)
    score1.metric(
        "Skill match",
        f'{results["skill_match_percentage"]:.2f}%'
    )
    score2.metric(
        "TF-IDF text similarity",
        f'{results["similarity_percentage"]:.2f}%'
    )

    st.caption(
        "These are project estimates based on detected skills and "
        "text similarity. They are not an employer's actual ATS score."
    )

    matched_col, missing_col = st.columns(2)

    with matched_col:
        st.subheader("✅ Matched skills")
        if results["matched_skills"]:
            for skill in results["matched_skills"]:
                st.write(f"• {skill}")
        else:
            st.info("No matching skills detected.")

    with missing_col:
        st.subheader("📌 Skills not detected in resume")
        if results["missing_skills"]:
            for skill in results["missing_skills"]:
                st.write(f"• {skill}")
        else:
            st.success("All detected job-description skills were found.")

    with st.expander("View all detected resume skills"):
        st.write(results["resume_skills"] or "No skills detected.")

    with st.expander("View extracted resume text"):
        st.text(results["extracted_resume_text"])

    # Step 3: Call Gemini only when the user requests AI feedback.
    st.divider()
    st.header("🤖 Gemini AI recommendations")
    st.write(
        "Your basic analysis is ready. Click below when you want Gemini "
        "to generate detailed feedback."
    )

    if st.button("✨ Generate AI recommendations"):
        try:
            with st.spinner("Gemini is preparing your feedback..."):
                ai = generate_ai_recommendations(
                    results["extracted_resume_text"],
                    st.session_state["analysis_job_description"],
                    results
                )
                st.session_state["ai_recommendations"] = ai
        except Exception as ai_error:
            st.session_state.pop("ai_recommendations", None)
            st.error("Gemini could not generate feedback this time.")
            st.caption(
                "Check the API key, model access, internet connection, "
                "and API quota."
            )
            st.code(str(ai_error))

    ai = st.session_state.get("ai_recommendations")

    if ai:
        sections = [
            ("💪 Resume strengths", "resume_strengths"),
            ("⚠️ Resume weaknesses", "resume_weaknesses"),
            ("📚 Missing skill recommendations", "missing_skill_recommendations"),
            ("💡 Improvement suggestions", "improvement_suggestions")
        ]

        for heading, key in sections:
            st.subheader(heading)
            items = ai.get(key, [])
            if isinstance(items, list) and items:
                for item in items:
                    st.write(f"• {item}")
            else:
                st.write("No points returned.")

        st.subheader("📝 Overall feedback")
        st.write(ai.get("overall_feedback", "No overall feedback returned."))

st.divider()
st.caption("SmartATS | Built with Python, NLP techniques, Gemini and Streamlit")


Overwriting app.py


In [8]:
# Stop any old process using Streamlit's port, then start the app again.
!fuser -k 8501/tcp >/dev/null 2>&1 || true
!nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true > /content/streamlit.log 2>&1 &


In [9]:
import time
import requests

for attempt in range(15):
    try:
        response = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=2)
        if response.status_code == 200:
            print("Streamlit is running:", response.text)
            break
    except requests.RequestException:
        time.sleep(2)
else:
    print("Streamlit did not respond. Check the log below.")
    print(open("/content/streamlit.log").read()[-4000:])

Streamlit is running: ok


In [10]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

cloudflared version 2026.11.1 (built 2026-11-17-15:04 UTC)


In [ ]:
!cloudflared tunnel --url http://127.0.0.1:8501

2026-11-19T08:38:21Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-19T08:38:21Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-19T08:38:26Z INF +--------------------------------------------------------------------------------------------+
2026-11-19T08:38:26Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-19T08:38:26Z INF |  https://hometown-floyd-secretary-effects.trycloudflar